# 四方法 SVG 检测流水线示例：`mouse_brain_STARmap`

本 Notebook 以 `mouse_brain_STARmap`（STARmap 小鼠大脑皮层，930 spots × 996 genes）为示例，
端到端演示本项目「批量空间可变基因（SVG）检测对比流水线」：

1. 读取数据集注册表（`configs/datasets.json`）与输入 h5ad；
2. 调用统一主控脚本 `src/pipeline/models_benchmark.sh` 完成 **共同前处理 → 四方法（SPARK-X / nnSVG / SpaGCN / SpaSEG）→ 统一评估**；
3. 汇总展示各方法的 SVG 排名、显著基因数、Moran's I、运行耗时与跨方法一致性指标。

> **运行环境**：Python 方法依赖 `envs/spatial`（见 `requirements.txt`），R 方法依赖
> `envs/spatial_R`（见 `renv.lock`），可用 `bash setup_linux_env.sh` 一键构建。
> 建议用 `envs/spatial/bin/python` 作为本 Notebook 的 kernel（已含 anndata / scanpy / matplotlib）。
> 项目整体说明见 [README.md](../README.md)。

## 0. 环境与路径初始化

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

# 项目根：Notebook 通常在 <root>/notebooks 目录下运行
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src   # 项目核心：数据集注册 / run 配置 / 共同前处理

DATASET = "mouse_brain_STARmap"
run = src.resolve_run(dataset=DATASET)

print("项目根    :", PROJECT_ROOT)
print("数据集    :", DATASET)
print("输入 h5ad :", run["h5ad"], "| 存在:", run["h5ad"].exists())
print("输出目录  :", run["outdir"])
print("样本标签  :", run["sample"])
print("技术类型  :", run["tech"], "| 维度:", run["dim"], "D")
print("可运行方法:", run["methods"])

## 1. 数据集注册表

`configs/datasets.json` 统一登记所有数据集的路径、规模与运行位置。本示例是小规模 2D
数据集（本地可全流程运行）；大规模数据（Visium HD / MERFISH 等）走 HPC 提交脚本。

In [ ]:
config = json.loads((PROJECT_ROOT / "configs" / "datasets.json").read_text(encoding="utf-8"))
print(json.dumps(config[DATASET], ensure_ascii=False, indent=2))

## 2. 输入数据一览

读取 h5ad：空间坐标在 `obsm['spatial']`，真实 counts 在 `layers['raw_count']`，
`obs['clusters']` 提供 9 个区域/细胞类型标签（供下游一致性评估使用）。

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

adata = ad.read_h5ad(run["h5ad"])
print(adata)
print("\nobs 列  :", list(adata.obs.columns))
print("layers  :", list(adata.layers.keys()))
print("obsm    :", list(adata.obsm.keys()))
print("clusters:", adata.obs["clusters"].nunique(), "类")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

coords = np.asarray(adata.obsm["spatial"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
cat = adata.obs["clusters"].astype(str).astype("category")
sca = axes[0].scatter(coords[:, 0], coords[:, 1], c=cat.cat.codes, s=4,
                      cmap="tab20", linewidths=0)
axes[0].set_title("区域标签 (clusters)")
sca2 = axes[1].scatter(coords[:, 0], coords[:, 1], c=adata.obs["total_counts"],
                       s=4, cmap="viridis", linewidths=0)
axes[1].set_title("UMI 总数")
for ax in axes:
    ax.set_aspect("equal")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## 3. 运行分析流水线

调用统一主控脚本 `src/pipeline/models_benchmark.sh`，自动完成：
**共同前处理**（导出 R 方法输入 + 各方法 h5ad）→ **依次运行四方法** → **统一评估与空间可视化**。

- `RUN_PIPELINE = True`：完整重跑（约 3–4 分钟，结果覆盖 `results/local_results/<dataset>/`）；
- 已有结果且只想复现展示时，可改为 `False` 直接读取现有结果。

In [ ]:
RUN_PIPELINE = True       # 重跑流水线；False 则直接读取现有结果
METHODS = "spark,nnsvg,spagcn,spaseg"
DEVICE = "auto"           # auto / cuda / cpu

In [ ]:
if RUN_PIPELINE:
    script = PROJECT_ROOT / "src" / "pipeline" / "models_benchmark.sh"
    cmd = ["bash", str(script), "--dataset", DATASET,
           "--methods", METHODS, "--device", DEVICE]
    env = dict(os.environ)
    if src.find_python():    env["SVG_PYTHON"]  = src.find_python()
    if src.find_rscript():   env["SVG_RSCRIPT"] = src.find_rscript()

    print("$ " + " ".join(cmd))
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env,
                          text=True, capture_output=True)
    print(proc.stdout[-8000:])           # 打印日志尾部（完整日志在各方法 logs/ 下）
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError("流水线运行失败")
    print(f"\n>>> 流水线完成，总耗时 {time.time() - t0:.1f}s")
else:
    print("RUN_PIPELINE=False，直接读取现有结果:", run["outdir"])

## 4. 各方法 SVG 排名结果

每个方法都输出统一格式的排名文件 `SVG_<method>_<sample>_rank.csv`
（列：`gene, stat, pval, padj, rank`，SpaGCN 额外含 `effect`），这里并表展示各方法
前 10 名 SVG。注：SpaGCN / SpaSEG 另有一份域对比后的候选显著基因表
（`SVG_<method>_<sample>.csv`），行数少于全基因排名。

In [ ]:
METHOD_LABEL = {"spark": "SPARK-X", "nnsvg": "nnSVG",
                "spagcn": "SpaGCN", "spaseg": "SpaSEG"}
METHOD_DIRS = {"spark": "SPARK_X", "nnsvg": "nnSVG",
               "spagcn": "spaGCN", "spaseg": "spaSEG"}

def load_ranking(method, outdir, sample):
    sub = Path(outdir) / METHOD_DIRS[method]
    # 优先统一排名文件 _rank.csv；缺失时回退主 CSV
    cands = sorted(sub.glob(f"SVG_*_{sample}_rank.csv"))
    if not cands:
        cands = sorted(sub.glob(f"SVG_*_{sample}.csv"))
    if not cands:
        return None
    df = pd.read_csv(cands[0])
    df = df.rename(columns={df.columns[0]: "gene"})
    if "rank" not in df.columns:          # 兜底：无 rank 列时按行序生成
        df["rank"] = np.arange(1, len(df) + 1)
    keep = [c for c in ("gene", "pval", "padj", "rank") if c in df.columns]
    df = df[keep].dropna(subset=["pval"]).reset_index(drop=True)
    df["method"] = METHOD_LABEL[method]
    return df

rankings = {m: load_ranking(m, run["outdir"], run["sample"]) for m in METHOD_DIRS}
rankings = {m: df for m, df in rankings.items() if df is not None}
assert rankings, f"未找到任何方法结果，请先运行第 3 节（{run['outdir']}）"

top10 = pd.concat(rankings.values(), ignore_index=True)
top10 = top10.sort_values(["method", "rank"]).groupby("method").head(10)
top10[["method", "rank", "gene", "pval", "padj"]].to_string(index=False)

## 5. 统一评估指标

`src/utils/evaluation.py` 针对同一 h5ad 汇总一套可比指标（显著基因数、Moran's I、
p-value 校准、区域一致性 ARI/NMI、运行耗时等），写入 `eval/summary.json`。

In [ ]:
eval_dir = run["outdir"] / "eval"
summary = json.loads((eval_dir / "summary.json").read_text(encoding="utf-8"))

rows = []
for m, st in summary["methods"].items():
    rows.append({
        "method": METHOD_LABEL.get(m, m),
        "n_sig": st["n_sig"],
        "n_ranked": st["n_ranked"],
        "median_moran_I": round(st["median_moran_I"], 4),
        "wall_seconds": round(st["wall_seconds"], 1),
        "ks_uniform_p": f"{st['ks_uniform_p']:.2e}",
        "ari_top100": round(st["ari_top100"], 4),
        "nmi_top100": round(st["nmi_top100"], 4),
    })
pd.DataFrame(rows).to_string(index=False)

## 6. 评估可视化

`evaluation.py` 与 `spatial_svg_plots.py` 会生成一批评估图（QQ 图、p-value 直方图、
排名一致性雷达图、跨方法一致性、top 基因空间表达图等），这里直接嵌入展示。

In [ ]:
from IPython.display import Image, display

fig_dir = eval_dir / "figures"

def show(rel, width=560):
    p = fig_dir / rel
    if p.exists():
        display(Image(filename=str(p), width=width))

for f in ["qq.png", "pval_hist.png", "radar.png", "rank_consensus.png"]:
    show(f)

In [ ]:
# 跨方法空间可视化（位于 figures/spatial/）
for f in ["cross_method_STARmap_Mouse_Brain.png",
          "top_expr_STARmap_Mouse_Brain.png",
          "pattern_gallery_STARmap_Mouse_Brain_spark.png"]:
    show("spatial/" + f, width=800)

## 7. 跨方法一致性小结

Kendall's W 描述四方法排名的总体一致性；`consensus_genes_top100` 给出被多方法共同认可
的高置信 SVG 基因（如髓鞘 / 神经元标记：`Mbp`、`Mobp`、`Slc17a7`、`Nrgn` 等）。
Spearman 相关矩阵展示两两方法排名相关性。

In [ ]:
cons = summary.get("consistency", {})
print("Kendall's W = %.3f" % cons.get("kendall_w", float("nan")))
print("\n共识 top20 SVG 基因:")
print(", ".join(cons.get("consensus_genes_top100", [])[:20]))

spearman = cons.get("spearman_matrix", {})
pd.DataFrame(spearman).round(3).to_string()

## 8. 下一步

- **换数据集**：把 `DATASET` 换成 `configs/datasets.json` 中的任意 key（如
  `Visium_Mouse_Olfactory_Bulb`、`DLPFC_151507`），即可复用同一流水线；
- **大数据集**（Visium HD / MERFISH / Slide-seq 3D）走 `src/pipeline/sbatch/` 的 HPC 提交脚本；
- **更多参数**：见 [README.md](../README.md) 与 `bash src/pipeline/models_benchmark.sh --help`。